<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Data Visualization**


Estimated time needed: **45** minutes


In this lab, you will focus on data visualization. The dataset will be provided through an RDBMS, and you will need to use SQL queries to extract the required data.


## Objectives


After completing this lab, you will be able to:


-   Visualize the distribution of data.

-   Visualize the relationship between two features.

-   Visualize composition and comparison of data.




## Demo: How to work with database


Download the database file.


In [4]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv

--2026-05-31 12:05:23--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
200 OKequest sent, awaiting response... 
Length: 159525875 (152M) [text/csv]
Saving to: ‘survey-data.csv’

survey-data.csv     100%[===================>] 152.13M  37.7MB/s    in 4.1s    

2026-05-31 12:05:34 (37.4 MB/s) - ‘survey-data.csv’ saved [159525875/159525875]



**Install and Import Necessary Python Libraries**

Ensure that you have the required libraries installed to work with SQLite and Pandas:


In [5]:
!pip install pandas 
!pip install matplotlib

import pandas as pd
import matplotlib.pyplot as plt

**Read the CSV File into a Pandas DataFrame**

Load the Stack Overflow survey data into a Pandas DataFrame:


In [ ]:
# Read the CSV file
df = pd.read_csv('survey-data.csv')

# Display the first few rows of the data
df.head()


**Create a SQLite Database and Insert the Data**

Now, let's create a new SQLite database (`survey-data.sqlite`) and insert the data from the DataFrame into a table using the sqlite3 library:


In [ ]:
import sqlite3

# Create a connection to the SQLite database
conn = sqlite3.connect('survey-data.sqlite')

# Write the dataframe to the SQLite database
df.to_sql('main', conn, if_exists='replace', index=False)


# Close the connection
conn.close()


**Verify the Data in the SQLite Database**
Verify that the data has been correctly inserted into the SQLite database by running a simple query:


In [ ]:
# Reconnect to the SQLite database
conn = sqlite3.connect('survey-data.sqlite')

# Run a simple query to check the data
QUERY = "SELECT * FROM main LIMIT 5"
df_check = pd.read_sql_query(QUERY, conn)

# Display the results
print(df_check)


## Demo: Running an SQL Query


Count the number of rows in the table named 'main'


In [ ]:
QUERY = """
SELECT COUNT(*) 
FROM main
"""
df = pd.read_sql_query(QUERY, conn)
df.head()


## Demo: Listing All Tables


To view the names of all tables in the database:


In [ ]:
QUERY = """
SELECT name as Table_Name FROM sqlite_master 
WHERE type = 'table'
"""
pd.read_sql_query(QUERY, conn)


## Demo: Running a Group By Query
    
For example, you can group data by a specific column, like Age, to get the count of respondents in each age group:


In [ ]:
QUERY = """
SELECT Age, COUNT(*) as count
FROM main
GROUP BY Age
ORDER BY Age
"""
pd.read_sql_query(QUERY, conn)


## Demo: Describing a table

Use this query to get the schema of a specific table, main in this case:


In [ ]:
table_name = 'main'

QUERY = """
SELECT sql FROM sqlite_master 
WHERE name= '{}'
""".format(table_name)

df = pd.read_sql_query(QUERY, conn)
print(df.iat[0,0])


## Hands-on Lab


### Visualizing the Distribution of Data

**Histograms**

Plot a histogram of CompTotal (Total Compensation).


In [ ]:
query = """
SELECT CompTotal
FROM main
WHERE CompTotal IS NOT NULL
"""

df_comp = pd.read_sql_query(query, conn)

plt.hist(df_comp["CompTotal"], bins=20)

plt.xlabel("Yearly Compensation")
plt.ylabel("Frequency")
plt.title("Histogram of ConvertedCompYearly")

plt.show()

**Box Plots**

Plot a box plot of Age.


In [ ]:
query = """
SELECT Age
FROM main
WHERE Age IS NOT NULL
"""

df_age = pd.read_sql_query(query, conn)

age_mapping = {
    "Under 18 years old": 17,
    "18-24 years old": 21,
    "25-34 years old": 30,
    "35-44 years old": 40,
    "45-54 years old": 50,
    "55-64 years old": 60,
    "65 years or older": 70
}

df_age["Age_numeric"] = df_age["Age"].map(age_mapping)

plt.figure(figsize=(8, 6))
plt.boxplot(df_age["Age_numeric"].dropna())

plt.ylabel("Age")
plt.title("Boxplot of Age")

plt.show()

### Visualizing Relationships in Data

**Scatter Plots**

Create a scatter plot of Age and WorkExp.


In [ ]:
query = """
SELECT Age, WorkExp
FROM main
WHERE Age IS NOT NULL
AND WorkExp IS NOT NULL
"""

df_scatter = pd.read_sql_query(query, conn)

age_order = {
    "Under 18 years old": 0,
    "18-24 years old": 1,
    "25-34 years old": 2,
    "35-44 years old": 3,
    "45-54 years old": 4,
    "55-64 years old": 5,
    "65 years or older": 6,
    "Prefer not to say": 7
}

df_scatter["age_order"] = df_scatter["Age"].map(age_order)

df_scatter = df_scatter.sort_values("age_order")

plt.figure(figsize=(15, 6))

plt.scatter(df_scatter["age_order"], df_scatter["WorkExp"])

plt.xlabel("Age")
plt.ylabel("Work Experience")
plt.title("Scatterplot of Age vs Work Experience")

plt.xticks(
    ticks=list(age_order.values()),
    labels=list(age_order.keys()),
    rotation=45
)

plt.show()

**Bubble Plots**

Create a bubble plot of `TimeSearching` and `Frustration` using the Age column as the bubble size.


In [ ]:
query = """
SELECT TimeSearching, Frustration, Age
FROM main
WHERE TimeSearching IS NOT NULL
AND Frustration IS NOT NULL
AND Age IS NOT NULL
"""

df_bubble = pd.read_sql_query(query, conn)
age_mapping = {
    "Under 18 years old": 17,
    "18-24 years old": 21,
    "25-34 years old": 30,
    "35-44 years old": 40,
    "45-54 years old": 50,
    "55-64 years old": 60,
    "65 years or older": 70,
    "Prefer not to say": 30
}

df_bubble["Age_numeric"] = df_bubble["Age"].map(age_mapping)

df_bubble["TimeSearching_code"] = pd.Categorical(df_bubble["TimeSearching"]).codes
df_bubble["Frustration_code"] = pd.Categorical(df_bubble["Frustration"]).codes

plt.figure(figsize=(12, 6))

plt.scatter(
    df_bubble["TimeSearching_code"],
    df_bubble["Frustration_code"],
    s=df_bubble["Age_numeric"] * 3,
    alpha=0.4
)

plt.xlabel("Time Searching")
plt.ylabel("Frustration")
plt.title("Bubble Plot of TimeSearching and Frustration by Age")

plt.show()

### Visualizing Composition of Data

**Pie Charts**

Create a pie chart of the top 5 databases(`DatabaseWantToWorkWith`) that respondents wish to learn next year.


In [ ]:
query = """
SELECT DatabaseWantToWorkWith
FROM main
WHERE DatabaseWantToWorkWith IS NOT NULL
"""

df_db = pd.read_sql_query(query, conn)

database_counts = (
    df_db["DatabaseWantToWorkWith"]
    .str.split(";")
    .explode()
    .value_counts()
    .head(5)
)

database_counts

plt.figure(figsize=(8, 8))

plt.pie(
    database_counts,
    labels=database_counts.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Top 5 Databases Respondents Want to Work With")

plt.show()

**Stacked Charts** 

Create a stacked bar chart of median `TimeSearching` and `TimeAnswering` for the age group 30 to 35.


In [ ]:
query = """
SELECT Age, TimeSearching, TimeAnswering
FROM main
WHERE Age IS NOT NULL
AND TimeSearching IS NOT NULL
AND TimeAnswering IS NOT NULL
"""

df_stack = pd.read_sql_query(query, conn)

df_stack = df_stack[df_stack["Age"] == "25-34 years old"]

time_mapping = {
    "Less than 15 minutes a day": 7.5,
    "15-30 minutes a day": 22.5,
    "30-60 minutes a day": 45,
    "60-120 minutes a day": 90,
    "Over 120 minutes a day": 150
}

df_stack["TimeSearching_numeric"] = df_stack["TimeSearching"].map(time_mapping)
df_stack["TimeAnswering_numeric"] = df_stack["TimeAnswering"].map(time_mapping)

median_searching = df_stack["TimeSearching_numeric"].median()
median_answering = df_stack["TimeAnswering_numeric"].median()

median_searching, median_answering

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

plt.bar("Age 30-35", median_searching, label="Time Searching")
plt.bar("Age 30-35", median_answering, bottom=median_searching, label="Time Answering")

plt.ylabel("Median Time, minutes per day")
plt.title("Median Time Searching and Answering for Age Group 30-35")
plt.legend()

plt.show()

### Visualizing Comparison of Data

**Line Chart**

Plot the median `CompTotal` for all ages from 45 to 60.


In [ ]:
query = """
SELECT Age, CompTotal
FROM main
WHERE Age IS NOT NULL
AND CompTotal IS NOT NULL
AND Age IN ('45-54 years old', '55-64 years old')
"""

df_line = pd.read_sql_query(query, conn)

median_comp = df_line.groupby("Age")["CompTotal"].median()
median_comp

age_order = ["45-54 years old", "55-64 years old"]

median_comp = median_comp.reindex(age_order)

plt.figure(figsize=(8, 6))

plt.plot(median_comp.index, median_comp.values, marker="o")

plt.xlabel("Age Group")
plt.ylabel("Median Total Compensation")
plt.title("Median CompTotal for Ages 45 to 60")

plt.show()

**Bar Chart**

Create a horizontal bar chart using the `MainBranch` column.


In [ ]:
query = """
SELECT MainBranch
FROM main
WHERE MainBranch IS NOT NULL
"""

df_branch = pd.read_sql_query(query, conn)
branch_counts = df_branch["MainBranch"].value_counts()

branch_counts

branch_counts = df_branch["MainBranch"].value_counts().sort_values()

plt.figure(figsize=(10, 6))

plt.barh(branch_counts.index, branch_counts.values)

plt.xlabel("Number of Respondents")
plt.ylabel("Main Branch")
plt.title("Distribution of MainBranch")

plt.tight_layout()
plt.show()

### Summary


In this lab, you focused on extracting and visualizing data from an RDBMS using SQL queries and SQLite. You applied various visualization techniques, including:

- Histograms to display the distribution of CompTotal.
- Box plots to show the spread of ages.
- Scatter plots and bubble plots to explore relationships between variables like Age, WorkExp, `TimeSearching` and `TimeAnswering`.
- Pie charts and stacked charts to visualize the composition of data.
- Line charts and bar charts to compare data across categories.


### Close the Database Connection

Once the lab is complete, ensure to close the database connection:


In [ ]:
conn.close()

## Authors:
Ayushi Jain


### Other Contributors:
- Rav Ahuja
- Lakshmi Holla
- Malika


Copyright © IBM Corporation. All rights reserved.
